# 1 下载google日历数据

In [11]:
source_ics_path = '../source_ics/google.ics'
custom_ics_path = './apple_supplement.ics'

In [4]:
import requests

# 要下载的网页 URL
url = 'https://calendar.google.com/calendar/ical/zh-cn.china%23holiday%40group.v.calendar.google.com/public/basic.ics'

# 发送 HTTP GET 请求
response = requests.get(url)

# 检查请求是否成功
if response.status_code == 200:
    # 将网页内容保存到文件中
    with open(source_ics_path, 'w', encoding='utf-8') as f:
        f.write(response.text)
    print("网页已成功下载并保存为 %s" % source_ics_path)
else:
    print(f"请求失败，状态码：{response.status_code}")


网页已成功下载并保存为 ../source_ics/google.ics


# 2 新建日历

In [54]:
from icalendar import Calendar

# 创建一个新的日历对象
custom_calendar = Calendar()

# 添加日历属性
custom_calendar.add('VERSION', '2.0')
custom_calendar.add('PRODID', 'icalendar-ruby')
custom_calendar.add('CALSCALE', 'GREGORIAN')
custom_calendar.add('X-WR-CALNAME', '中国大陆节假日')
custom_calendar.add('X-APPLE-LANGUAGE', 'zh')
custom_calendar.add('X-APPLE-REGION', 'CN')

# 将内容写入 .ics 文件
with open(custom_ics_path, 'wb') as f:
    f.write(custom_calendar.to_ical())

print("创建新的的 .ics 文件已保存为 %s" % custom_ics_path)


创建新的的 .ics 文件已保存为 ./apple_supplement.ics


# 2 读取google日历数据
读取google日历数据，摘取其中 '教师节' 事件

In [57]:
from icalendar import Calendar, Event
import hashlib

# 打开并读取 .ics 文件
with open(source_ics_path, 'r', encoding='utf-8') as f:
    source_calendar = Calendar.from_ical(f.read())
with open(custom_ics_path, 'r', encoding='utf-8') as f:
    custom_calendar = Calendar.from_ical(f.read())

# 遍历事件并打印详情
for event in source_calendar.walk('VEVENT'):
    if event.get('SUMMARY') == '教师节':
        new_event = Event()
        new_event.add('SUMMARY', event.get('SUMMARY'))
        new_event.add('DTSTART;VALUE=DATE', event.get('DTSTART').dt)
        new_event.add('DTEND;VALUE=DATE', event.get('DTEND').dt)
        new_event.add('TRANSP', event.get('TRANSP'))
        new_event.add('UID', f"{hashlib.md5(str(event).encode()).hexdigest()}@jayden")
        custom_calendar.add_component(new_event)

# 保存修改后的 .ics 文件
with open(custom_ics_path, 'wb') as f:
    f.write(custom_calendar.to_ical())
